# Notebook 12: Hands-On Lab — End-to-End Customer Segmentation

**Difficulty:** ⭐⭐⭐⭐ | **Estimated Time:** 2.5 hours

---

## Lab Overview

This is a full-pipeline, business-driven data science lab. You will:

1. **Generate** a realistic synthetic e-commerce dataset (Online Retail style, 4,000 customers)
2. **Explore** the data through EDA: pairplots, distributions, correlations, outlier detection
3. **Engineer features**: RFM (Recency, Frequency, MonetaryValue) + basket size + product diversity
4. **Preprocess**: handle skewness (log transforms) and scale
5. **Select k**: elbow, silhouette, Davies-Bouldin — three methods, one conclusion
6. **Cluster**: KMeans, Hierarchical (Ward), DBSCAN — compare and contrast
7. **Reduce dimensions**: PCA, t-SNE, and optionally UMAP — for visualisation and interpretation
8. **Soft assignments**: GMM for uncertainty-aware segmentation
9. **Detect anomalies**: Isolation Forest to flag suspicious customers
10. **Deliver a business report**: actionable segment descriptions, campaign recommendations, model persistence

---

**Week 11 theme:** Retail customer segmentation — turning raw transaction data into marketing strategy

## Business Context

You are a data scientist at **ShopStream**, a mid-sized UK e-commerce retailer selling home goods. The Head of Marketing has sent you the following brief:

> *"We have about 4,000 active customers but we're sending them all the same weekly newsletter. Conversion is terrible — 1.2%. I know we have very different types of customers: some buy every week, some haven't bought in months, some spend £500 per order, others spend £8. We need to talk to each group differently."*

**Requested outputs:**

| Segment type | Suspected campaign |
|---|---|
| Loyal high-value customers | VIP early access, premium support, referral program |
| At-risk dormant customers | Win-back: 20% discount, "We miss you" email |
| New customers | Welcome series: onboarding, tutorials, first-purchase incentive |
| Occasional / mid-tier | Upsell: cross-sell based on category history |

**Our job:** Find these segments from purchase data using unsupervised learning, validate the segmentation, and hand the marketing team a segment-per-customer lookup table they can load into their CRM.

In [ ]:
# ── Core libraries ────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')   # suppress convergence / deprecation noise

# ── Visualisation ─────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from scipy.stats import zscore

# ── Clustering ────────────────────────────────────────────────────────────────
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.mixture import GaussianMixture
from sklearn.ensemble import IsolationForest

# ── Dimensionality reduction ──────────────────────────────────────────────────
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

# ── Preprocessing and evaluation ─────────────────────────────────────────────
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (silhouette_score, davies_bouldin_score,
                              adjusted_rand_score, confusion_matrix)
from sklearn.datasets import make_blobs
import joblib   # for model persistence

# ── Global plot style ─────────────────────────────────────────────────────────
sns.set_theme(style='whitegrid')
np.random.seed(42)

print('All imports successful.')

# Optional UMAP (will be handled gracefully if not installed)
try:
    import umap
    UMAP_AVAILABLE = True
    print('umap-learn is available.')
except ImportError:
    UMAP_AVAILABLE = False
    print('umap-learn not installed — will use t-SNE instead in that section.')

In [ ]:
np.random.seed(42)

# ════════════════════════════════════════════════════════════════════════════
#  Generate Realistic Synthetic Online Retail Dataset — 4,000 Customers
# ════════════════════════════════════════════════════════════════════════════
# We simulate 4 distinct customer segments with realistic RFM profiles,
# then add noise to make the problem realistic.

N = 4000   # total customers

# Segment sizes (roughly equal with some imbalance)
seg_sizes = [900, 1100, 1200, 800]   # VIP, At-Risk, New, Occasional

# Segment 0 — VIP / High-Value Loyal
#   Low recency (bought recently), high frequency, very high monetary
r0 = np.random.normal(15,  8,  seg_sizes[0]).clip(1, 60)    # recency in days
f0 = np.random.normal(20,  5,  seg_sizes[0]).clip(5, 50)    # orders per year
m0 = np.random.lognormal(6.5, 0.4, seg_sizes[0])            # monetary ~£665 avg
b0 = np.random.normal(4.5, 1.0, seg_sizes[0]).clip(1, 12)   # avg basket size
d0 = np.random.normal(8,   2,   seg_sizes[0]).clip(1, 15)   # product diversity

# Segment 1 — At-Risk / Dormant
#   High recency (haven't bought in a while), low frequency, low-medium monetary
r1 = np.random.normal(180, 40,  seg_sizes[1]).clip(60, 365) # recency in days
f1 = np.random.normal(3,   1.5, seg_sizes[1]).clip(1, 10)
m1 = np.random.lognormal(4.2, 0.5, seg_sizes[1])            # monetary ~£67 avg
b1 = np.random.normal(2.5, 0.8, seg_sizes[1]).clip(1, 8)
d1 = np.random.normal(3,   1.5, seg_sizes[1]).clip(1, 8)

# Segment 2 — New Customers
#   Very low recency (just joined), low frequency (only 1-2 orders), variable spend
r2 = np.random.normal(10,  5,   seg_sizes[2]).clip(1, 30)
f2 = np.random.normal(1.5, 0.7, seg_sizes[2]).clip(1, 5)
m2 = np.random.lognormal(4.5, 0.6, seg_sizes[2])            # monetary ~£90 avg
b2 = np.random.normal(3,   1.0, seg_sizes[2]).clip(1, 8)
d2 = np.random.normal(2,   1.0, seg_sizes[2]).clip(1, 6)

# Segment 3 — Occasional / Mid-Tier
#   Medium recency, medium frequency, medium spend
r3 = np.random.normal(60,  20,  seg_sizes[3]).clip(10, 180)
f3 = np.random.normal(6,   2,   seg_sizes[3]).clip(2, 15)
m3 = np.random.lognormal(5.0, 0.5, seg_sizes[3])            # monetary ~£148 avg
b3 = np.random.normal(3.5, 1.0, seg_sizes[3]).clip(1, 10)
d3 = np.random.normal(5,   2,   seg_sizes[3]).clip(1, 12)

# Stack all segments
recency     = np.concatenate([r0, r1, r2, r3])
frequency   = np.concatenate([f0, f1, f2, f3])
monetary    = np.concatenate([m0, m1, m2, m3])
basket_size = np.concatenate([b0, b1, b2, b3])
diversity   = np.concatenate([d0, d1, d2, d3])
true_labels = np.concatenate([
    np.zeros(seg_sizes[0]),   # 0 = VIP
    np.ones(seg_sizes[1]),    # 1 = At-Risk
    np.full(seg_sizes[2], 2), # 2 = New
    np.full(seg_sizes[3], 3)  # 3 = Occasional
]).astype(int)

# Build DataFrame with customer IDs
customer_ids = [f'CUST_{i:05d}' for i in range(1, N + 1)]
df = pd.DataFrame({
    'CustomerID':      customer_ids,
    'Recency':         recency.round(0).astype(int),
    'Frequency':       frequency.round(1),
    'MonetaryValue':   monetary.round(2),
    'AvgBasketSize':   basket_size.round(1),
    'ProductDiversity':diversity.round(0).astype(int),
    '_true_segment':   true_labels    # underscore = internal, not shown to model
})

# Shuffle so segments aren't in order
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

print(f'Dataset shape: {df.shape}')
print(f'\nFirst 5 rows:')
print(df.drop('_true_segment', axis=1).head())
print(f'\nSegment distribution (true):')
seg_names = {0: 'VIP', 1: 'At-Risk', 2: 'New', 3: 'Occasional'}
for k, v in seg_names.items():
    n = (df['_true_segment'] == k).sum()
    print(f'  Segment {k} ({v}): {n} customers ({n/N:.1%})')

In [ ]:
np.random.seed(42)
sns.set_theme(style='whitegrid')

# ── Exploratory Data Analysis ────────────────────────────────────────────────
# Feature columns only (exclude CustomerID and true_segment)
feature_cols = ['Recency', 'Frequency', 'MonetaryValue', 'AvgBasketSize', 'ProductDiversity']
X_raw = df[feature_cols].values

print('=== Descriptive Statistics ===')
print(df[feature_cols].describe().round(2))

# Pairplot — coloured by true segment to validate our data generation
# (in practice, we would NOT have this colouring)
pairplot_df = df[feature_cols + ['_true_segment']].copy()
pairplot_df['Segment'] = pairplot_df['_true_segment'].map(seg_names)
g = sns.pairplot(
    pairplot_df.drop('_true_segment', axis=1),
    hue='Segment',
    plot_kws={'alpha': 0.3, 's': 10},
    diag_kind='kde',
    palette=['#2196F3', '#FF5722', '#4CAF50', '#9C27B0']
)
g.fig.suptitle('Pairplot of All RFM Features (coloured by true segment for validation)',
               y=1.02, fontsize=12)
plt.show()
print('Key observation: MonetaryValue and Recency show the clearest segment separation.')

In [ ]:
np.random.seed(42)
sns.set_theme(style='whitegrid')

# ── Distribution histograms and skewness check ───────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.ravel()

for i, col in enumerate(feature_cols):
    axes[i].hist(df[col], bins=50, color='steelblue', alpha=0.7, edgecolor='white')
    skew_val = df[col].skew()
    axes[i].set_title(f'{col}\nSkewness: {skew_val:.2f}')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Count')

# Hide the unused 6th subplot
axes[5].set_visible(False)

plt.suptitle('Feature Distributions — Raw Data', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

# Correlation heatmap
fig, ax = plt.subplots(figsize=(7, 5))
corr_matrix = df[feature_cols].corr()
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, square=True, ax=ax, linewidths=0.5)
ax.set_title('Feature Correlation Matrix', fontsize=13)
plt.tight_layout()
plt.show()

print('Key finding: MonetaryValue is right-skewed (skew > 1) — will need log-transform.')
print('Frequency and AvgBasketSize are moderately correlated — expected (bigger baskets, more orders).')

In [ ]:
np.random.seed(42)
sns.set_theme(style='whitegrid')

# ── Flag potential outliers visually using box plots ─────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, col in zip(axes, ['MonetaryValue', 'Recency', 'Frequency']):
    ax.boxplot(df[col], vert=True, patch_artist=True,
               boxprops=dict(facecolor='lightsteelblue'),
               medianprops=dict(color='crimson', linewidth=2),
               flierprops=dict(marker='o', alpha=0.3, markersize=3, color='grey'))
    # Count IQR outliers
    Q1, Q3 = df[col].quantile([0.25, 0.75])
    IQR     = Q3 - Q1
    n_out   = ((df[col] < Q1 - 1.5 * IQR) | (df[col] > Q3 + 1.5 * IQR)).sum()
    ax.set_title(f'{col}\n(IQR outliers: {n_out})')
    ax.set_ylabel(col)

plt.suptitle('Box Plots — Potential Outliers in Key Features', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

# Identify top 5 biggest spenders as a sense-check
print('Top 5 customers by MonetaryValue:')
print(df[['CustomerID', 'MonetaryValue', 'Frequency', 'Recency']]
      .nlargest(5, 'MonetaryValue').to_string(index=False))

In [ ]:
np.random.seed(42)
sns.set_theme(style='whitegrid')

# ════════════════════════════════════════════════════════════════════════════
#  Data Preprocessing
#  1. Log-transform right-skewed features
#  2. StandardScaler
#  3. Show before/after distribution
# ════════════════════════════════════════════════════════════════════════════

df_proc = df[feature_cols].copy()

# Log-transform skewed features (log1p handles zeros safely)
# MonetaryValue is strongly right-skewed; Recency less so but still benefits
skewed_cols = ['MonetaryValue', 'Recency', 'Frequency']
for col in skewed_cols:
    df_proc[col] = np.log1p(df_proc[col])   # log(1 + x) is safe for x >= 0

# StandardScaler: zero mean, unit variance
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_proc)   # numpy array for sklearn

# ── Before vs. after for MonetaryValue ──────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

axes[0, 0].hist(df['MonetaryValue'], bins=60, color='salmon', alpha=0.8, edgecolor='white')
axes[0, 0].set_title('MonetaryValue — Raw\n(heavily right-skewed)')
axes[0, 0].set_xlabel('£ value')

axes[0, 1].hist(np.log1p(df['MonetaryValue']), bins=60, color='steelblue', alpha=0.8, edgecolor='white')
axes[0, 1].set_title('MonetaryValue — Log Transformed\n(approximately normal)')
axes[0, 1].set_xlabel('log(1 + £ value)')

axes[1, 0].hist(df['Recency'], bins=60, color='salmon', alpha=0.8, edgecolor='white')
axes[1, 0].set_title('Recency — Raw')
axes[1, 0].set_xlabel('Days since last purchase')

axes[1, 1].hist(np.log1p(df['Recency']), bins=60, color='steelblue', alpha=0.8, edgecolor='white')
axes[1, 1].set_title('Recency — Log Transformed')
axes[1, 1].set_xlabel('log(1 + days)')

plt.suptitle('Log Transformation Reduces Skewness', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

print(f'Scaled data shape: {X_scaled.shape}')
print(f'Mean (should be ~0): {X_scaled.mean(axis=0).round(3)}')
print(f'Std  (should be ~1): {X_scaled.std(axis=0).round(3)}')

In [ ]:
np.random.seed(42)
sns.set_theme(style='whitegrid')

# ════════════════════════════════════════════════════════════════════════════
#  Determine Optimal k — Three Methods
#  1. Elbow (inertia): look for the "elbow" where adding more clusters yields
#     diminishing returns in within-cluster sum of squares.
#  2. Silhouette score: measures how tight and well-separated clusters are.
#     Higher is better. Range: [-1, 1].
#  3. Davies-Bouldin index: ratio of within-cluster scatter to between-cluster
#     separation. Lower is better. Range: [0, infinity).
# ════════════════════════════════════════════════════════════════════════════

k_range    = range(2, 11)
inertias   = []
silhouettes= []
db_scores  = []

print('Fitting KMeans for k = 2 to 10...')
for k in k_range:
    km = KMeans(n_clusters=k, n_init=15, random_state=42)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, labels, sample_size=2000, random_state=42))
    db_scores.append(davies_bouldin_score(X_scaled, labels))
    print(f'  k={k}: inertia={km.inertia_:.0f}  sil={silhouettes[-1]:.3f}  DB={db_scores[-1]:.3f}')

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Elbow plot
axes[0].plot(list(k_range), inertias, 'o-', color='steelblue', lw=2)
axes[0].axvline(4, color='crimson', linestyle='--', label='k=4 (chosen)')
axes[0].set_title('Elbow Method — Inertia\n(look for the kink)')
axes[0].set_xlabel('Number of clusters (k)')
axes[0].set_ylabel('Inertia (WCSS)')
axes[0].legend()

# Silhouette
axes[1].plot(list(k_range), silhouettes, 'o-', color='forestgreen', lw=2)
axes[1].axvline(4, color='crimson', linestyle='--', label='k=4 (chosen)')
axes[1].set_title('Silhouette Score\n(higher is better)')
axes[1].set_xlabel('Number of clusters (k)')
axes[1].set_ylabel('Silhouette score')
axes[1].legend()

# Davies-Bouldin
axes[2].plot(list(k_range), db_scores, 'o-', color='darkorange', lw=2)
axes[2].axvline(4, color='crimson', linestyle='--', label='k=4 (chosen)')
axes[2].set_title('Davies-Bouldin Index\n(lower is better)')
axes[2].set_xlabel('Number of clusters (k)')
axes[2].set_ylabel('DB score')
axes[2].legend()

plt.suptitle('Optimal k Selection — All Three Methods Suggest k = 4', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print('\nConclusion: k=4 is optimal — matches all three metrics and business context (4 campaign types).')

In [ ]:
np.random.seed(42)
sns.set_theme(style='whitegrid')

# ── Apply KMeans(k=4) ────────────────────────────────────────────────────────
kmeans = KMeans(n_clusters=4, n_init=20, random_state=42)
kmeans_labels = kmeans.fit_predict(X_scaled)

# Store in DataFrame
df['KMeans_Cluster'] = kmeans_labels

# ── PCA to 2D for visualisation ──────────────────────────────────────────────
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)
df['PC1'] = X_pca[:, 0]
df['PC2'] = X_pca[:, 1]

explained = pca.explained_variance_ratio_
print(f'PCA: PC1 explains {explained[0]:.1%}, PC2 explains {explained[1]:.1%} '
      f'(total: {sum(explained):.1%})')

# ── Plot KMeans clusters in PCA space ────────────────────────────────────────
palette = ['#2196F3', '#FF5722', '#4CAF50', '#9C27B0']
cluster_names_kmeans = {}

fig, ax = plt.subplots(figsize=(9, 7))
for c in range(4):
    mask = kmeans_labels == c
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1],
               color=palette[c], alpha=0.4, s=15,
               label=f'Cluster {c} (n={mask.sum()})')

# Mark cluster centroids in PCA space
centroids_pca = pca.transform(kmeans.cluster_centers_)
ax.scatter(centroids_pca[:, 0], centroids_pca[:, 1],
           c='black', marker='*', s=300, zorder=5, label='Centroids')

ax.set_title('KMeans (k=4) — Clusters in PCA Space', fontsize=13)
ax.set_xlabel(f'PC1 ({explained[0]:.1%} variance)')
ax.set_ylabel(f'PC2 ({explained[1]:.1%} variance)')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
np.random.seed(42)
sns.set_theme(style='whitegrid')

# ════════════════════════════════════════════════════════════════════════════
#  Cluster Profiling — Who is in each KMeans cluster?
# ════════════════════════════════════════════════════════════════════════════

# Compute per-cluster mean for original (unscaled, untransformed) features
profile_cols = feature_cols + ['KMeans_Cluster']
cluster_profile = (df[profile_cols]
                   .groupby('KMeans_Cluster')
                   .mean()
                   .round(1))
cluster_profile['Count'] = df.groupby('KMeans_Cluster').size()

# Identify segment type based on Recency and MonetaryValue
# Low Recency + High Monetary = VIP
# High Recency + Low Monetary = At-Risk
# Very Low Recency + Low Frequency = New
# Medium everything = Occasional
segment_map = {}
for c in range(4):
    row = cluster_profile.loc[c]
    if row['Recency'] < 30 and row['MonetaryValue'] > 300:
        segment_map[c] = 'VIP / High-Value'
    elif row['Recency'] > 100:
        segment_map[c] = 'At-Risk / Dormant'
    elif row['Frequency'] < 3:
        segment_map[c] = 'New Customers'
    else:
        segment_map[c] = 'Occasional / Mid-Tier'

cluster_profile['Segment'] = [segment_map[c] for c in cluster_profile.index]

# Display styled table
print('=== KMeans Cluster Profiles ===')
print(cluster_profile[['Segment', 'Recency', 'Frequency', 'MonetaryValue',
                         'AvgBasketSize', 'ProductDiversity', 'Count']].to_string())

# Store the mapping for later use
df['Segment_Name'] = df['KMeans_Cluster'].map(segment_map)

In [ ]:
np.random.seed(42)
sns.set_theme(style='whitegrid')

# ════════════════════════════════════════════════════════════════════════════
#  Radar / Spider Chart — One per Cluster
#  Excellent for business presentations: shows each segment's "shape"
# ════════════════════════════════════════════════════════════════════════════

# Normalise features to [0, 1] for radar chart
radar_features = ['Recency', 'Frequency', 'MonetaryValue', 'AvgBasketSize', 'ProductDiversity']

cluster_means   = df.groupby('KMeans_Cluster')[radar_features].mean()
col_min         = cluster_means.min()
col_max         = cluster_means.max()
# Invert Recency: lower recency is BETTER for a customer, so we flip it
cluster_normed  = (cluster_means - col_min) / (col_max - col_min + 1e-9)
cluster_normed['Recency'] = 1 - cluster_normed['Recency']   # flip so high = recently active

# Radar chart setup
N_feat  = len(radar_features)
angles  = [n / float(N_feat) * 2 * np.pi for n in range(N_feat)]
angles += angles[:1]   # close the loop
labels  = ['Recent\n(inverted)', 'Frequency', 'Monetary', 'Basket', 'Diversity']

fig, axes = plt.subplots(2, 2, figsize=(12, 10), subplot_kw=dict(polar=True))
axes = axes.ravel()

for c, ax in enumerate(axes):
    values  = cluster_normed.loc[c].tolist()
    values += values[:1]   # close the loop

    ax.plot(angles, values, 'o-', linewidth=2, color=palette[c])
    ax.fill(angles, values, alpha=0.25, color=palette[c])
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(labels, fontsize=9)
    ax.set_ylim(0, 1)
    ax.set_title(f'Cluster {c} — {segment_map.get(c, "Unknown")}\n'
                 f'(n={df["KMeans_Cluster"].eq(c).sum()})',
                 fontsize=10, pad=20)
    ax.set_yticks([0.25, 0.5, 0.75])
    ax.set_yticklabels(['0.25', '0.5', '0.75'], fontsize=7, color='grey')

plt.suptitle('Radar Chart — Customer Segment Profiles\n'
             '(normalised; "Recent" is inverted so high = recently active)',
             fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
np.random.seed(42)
sns.set_theme(style='whitegrid')

# ════════════════════════════════════════════════════════════════════════════
#  Hierarchical Clustering — Ward Linkage
# ════════════════════════════════════════════════════════════════════════════
# Ward linkage minimises the total within-cluster variance when merging.
# It tends to produce compact, roughly equal-sized clusters.

# Compute linkage matrix on a sample (full 4000-point dendrogram is too large to read)
sample_idx = np.random.choice(N, 300, replace=False)
X_hc_sample = X_scaled[sample_idx]

Z = linkage(X_hc_sample, method='ward')   # Z is the linkage matrix

fig, ax = plt.subplots(figsize=(16, 5))
dendrogram(
    Z,
    ax=ax,
    truncate_mode='level',   # only show last few merges
    p=5,
    leaf_rotation=90,
    leaf_font_size=8,
    color_threshold=0.7 * max(Z[:, 2])   # colour by height threshold
)
ax.set_title('Ward Linkage Dendrogram (300-customer sample)\nHorizontal red line suggests k=4',
             fontsize=12)
ax.set_xlabel('Customer index (sample)')
ax.set_ylabel('Linkage distance')
# Draw a horizontal line suggesting the k=4 cut
ax.axhline(y=np.sort(Z[:, 2])[-3] * 0.9, color='crimson', linestyle='--',
           label='Suggested cut (k=4)')
ax.legend()
plt.tight_layout()
plt.show()

# Apply to full dataset using sklearn AgglomerativeClustering
agg = AgglomerativeClustering(n_clusters=4, linkage='ward')
agg_labels = agg.fit_predict(X_scaled)
df['HC_Cluster'] = agg_labels

# Compare with KMeans using Adjusted Rand Index
ari_km_hc = adjusted_rand_score(kmeans_labels, agg_labels)
print(f'Adjusted Rand Index (KMeans vs Hierarchical): {ari_km_hc:.3f}')
print('ARI = 1.0 means perfect agreement; ARI = 0.0 means random. '
      f'ARI = {ari_km_hc:.3f} suggests the two methods largely agree.')

In [ ]:
np.random.seed(42)
sns.set_theme(style='whitegrid')

# ════════════════════════════════════════════════════════════════════════════
#  DBSCAN — Density-Based Clustering
#  Finds natural clusters of arbitrary shape AND labels noise points
# ════════════════════════════════════════════════════════════════════════════
# Step 1: Use k-distance graph to select epsilon
# The "elbow" in the k-distance graph is the ideal epsilon

from sklearn.neighbors import NearestNeighbors

# Compute distances to 5th nearest neighbour for each point
k_nn = 5
nbrs = NearestNeighbors(n_neighbors=k_nn).fit(X_scaled)
distances, _ = nbrs.kneighbors(X_scaled)
k_distances  = np.sort(distances[:, k_nn - 1])   # distance to k-th neighbour, sorted

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(k_distances, color='steelblue', lw=1.5)
ax.axhline(y=1.2, color='crimson', linestyle='--', label='Selected epsilon = 1.2')
ax.set_title('k-Distance Graph (k=5)\nChoose epsilon at the "elbow"', fontsize=12)
ax.set_xlabel('Points sorted by distance to 5th neighbour')
ax.set_ylabel('Distance to 5th nearest neighbour')
ax.legend()
plt.tight_layout()
plt.show()

# Apply DBSCAN with chosen epsilon
dbscan = DBSCAN(eps=1.2, min_samples=15)
dbscan_labels = dbscan.fit_predict(X_scaled)   # -1 = noise
df['DBSCAN_Cluster'] = dbscan_labels

n_clusters_db = len(set(dbscan_labels)) - (1 if -1 in dbscan_labels else 0)
n_noise       = (dbscan_labels == -1).sum()
print(f'DBSCAN found {n_clusters_db} clusters and {n_noise} noise points '
      f'({n_noise/N:.1%} of customers).')
print(f'Noise points = potential outlier customers worth investigating.')

In [ ]:
np.random.seed(42)
sns.set_theme(style='whitegrid')

# ── Three-way comparison: KMeans vs. Hierarchical vs. DBSCAN ─────────────────

# ARI between all pairs
ari_km_hc   = adjusted_rand_score(kmeans_labels,  agg_labels)
ari_km_db   = adjusted_rand_score(kmeans_labels,  dbscan_labels)
ari_hc_db   = adjusted_rand_score(agg_labels,     dbscan_labels)

print('=== Adjusted Rand Index (ARI) — Higher is Better ===')
print(f'  KMeans  vs. Hierarchical : {ari_km_hc:.3f}')
print(f'  KMeans  vs. DBSCAN       : {ari_km_db:.3f}')
print(f'  Hierarch vs. DBSCAN      : {ari_hc_db:.3f}')

# Contingency matrix: KMeans vs Hierarchical
# Only use non-noise DBSCAN points for clean comparison
mask_valid = dbscan_labels != -1
cm_matrix = confusion_matrix(kmeans_labels[mask_valid], agg_labels[mask_valid])

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm_matrix, annot=True, fmt='d', cmap='Blues',
            xticklabels=[f'HC-{i}' for i in range(4)],
            yticklabels=[f'KM-{i}' for i in range(4)],
            ax=ax, linewidths=0.5)
ax.set_title('Contingency Matrix: KMeans vs. Hierarchical Clustering\n'
             '(near-diagonal = strong agreement)', fontsize=11)
ax.set_xlabel('Hierarchical Cluster')
ax.set_ylabel('KMeans Cluster')
plt.tight_layout()
plt.show()

In [ ]:
np.random.seed(42)
sns.set_theme(style='whitegrid')

# ── Side-by-side PCA visualisation: KMeans, HC, DBSCAN ──────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

all_labels   = [kmeans_labels, agg_labels, dbscan_labels]
titles       = ['KMeans (k=4)', 'Hierarchical Ward (k=4)', 'DBSCAN (ε=1.2, min_samples=15)']
all_palettes = [palette, palette, ['#2196F3', '#FF5722', '#4CAF50', '#9C27B0', 'grey']]

for ax, labels, title, pal in zip(axes, all_labels, titles, all_palettes):
    unique_labels = sorted(set(labels))
    for i, lbl in enumerate(unique_labels):
        mask = labels == lbl
        color = 'lightgrey' if lbl == -1 else pal[i % len(pal)]
        name  = 'Noise' if lbl == -1 else f'Cluster {lbl}'
        ax.scatter(X_pca[mask, 0], X_pca[mask, 1],
                   color=color, alpha=0.4, s=12, label=f'{name} (n={mask.sum()})')
    ax.set_title(title, fontsize=11)
    ax.set_xlabel(f'PC1 ({explained[0]:.1%})')
    ax.set_ylabel(f'PC2 ({explained[1]:.1%})')
    ax.legend(fontsize=7, markerscale=2)

plt.suptitle('Three Clustering Algorithms in PCA Space', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
np.random.seed(42)
sns.set_theme(style='whitegrid')

# ════════════════════════════════════════════════════════════════════════════
#  t-SNE Visualisation — Captures Non-Linear Structure
#  t-SNE is a non-linear dimensionality reduction that preserves
#  local neighbourhood structure. Great for visualisation, NOT for inference.
#  WARNING: t-SNE distances between clusters are NOT meaningful.
# ════════════════════════════════════════════════════════════════════════════

# Subsample for speed (t-SNE is O(n^2))
tsne_sample_size = min(2000, N)
tsne_idx  = np.random.choice(N, tsne_sample_size, replace=False)
X_tsne_in = X_scaled[tsne_idx]

print(f'Running t-SNE on {tsne_sample_size} customers (perplexity=30)...')
tsne = TSNE(n_components=2, perplexity=30, learning_rate='auto',
            init='pca', random_state=42, n_iter=1000)
X_tsne = tsne.fit_transform(X_tsne_in)
print('t-SNE complete.')

fig, ax = plt.subplots(figsize=(9, 7))
km_labels_sub = kmeans_labels[tsne_idx]
for c in range(4):
    mask = km_labels_sub == c
    ax.scatter(X_tsne[mask, 0], X_tsne[mask, 1],
               color=palette[c], alpha=0.5, s=15,
               label=f'Cluster {c} — {segment_map.get(c,"?")} (n={mask.sum()})')

ax.set_title('t-SNE Visualisation (2,000 customers)\nColoured by KMeans cluster',
             fontsize=13)
ax.set_xlabel('t-SNE 1')
ax.set_ylabel('t-SNE 2')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

print('\nNote: t-SNE shows clear cluster separation, confirming our k=4 segmentation is robust.')
print('Remember: distances BETWEEN clusters in t-SNE space are not meaningful — only local structure is preserved.')

In [ ]:
np.random.seed(42)
sns.set_theme(style='whitegrid')

# ════════════════════════════════════════════════════════════════════════════
#  UMAP vs t-SNE vs PCA — Three Projections
#  UMAP preserves both local AND global structure better than t-SNE
# ════════════════════════════════════════════════════════════════════════════

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. PCA (already computed)
X_pca_sub = X_pca[tsne_idx]
for c in range(4):
    mask = km_labels_sub == c
    axes[0].scatter(X_pca_sub[mask, 0], X_pca_sub[mask, 1],
                    color=palette[c], alpha=0.4, s=12,
                    label=f'C{c} — {segment_map.get(c,"?")}')
axes[0].set_title(f'PCA (PC1+PC2 = {sum(explained):.1%} variance)')
axes[0].set_xlabel('PC1')
axes[0].set_ylabel('PC2')
axes[0].legend(fontsize=7)

# 2. t-SNE (already computed)
for c in range(4):
    mask = km_labels_sub == c
    axes[1].scatter(X_tsne[mask, 0], X_tsne[mask, 1],
                    color=palette[c], alpha=0.4, s=12)
axes[1].set_title('t-SNE (perplexity=30)\nLocal structure preserved')
axes[1].set_xlabel('t-SNE 1')
axes[1].set_ylabel('t-SNE 2')

# 3. UMAP (if available) or second t-SNE with different perplexity
if UMAP_AVAILABLE:
    reducer = umap.UMAP(n_components=2, n_neighbors=30, min_dist=0.1, random_state=42)
    X_umap  = reducer.fit_transform(X_tsne_in)
    for c in range(4):
        mask = km_labels_sub == c
        axes[2].scatter(X_umap[mask, 0], X_umap[mask, 1],
                        color=palette[c], alpha=0.4, s=12)
    axes[2].set_title('UMAP (n_neighbors=30)\nLocal + global structure preserved')
    axes[2].set_xlabel('UMAP 1')
    axes[2].set_ylabel('UMAP 2')
else:
    # Fallback: t-SNE with higher perplexity
    tsne_high = TSNE(n_components=2, perplexity=50, learning_rate='auto',
                     init='pca', random_state=42, n_iter=800)
    X_tsne_high = tsne_high.fit_transform(X_tsne_in)
    for c in range(4):
        mask = km_labels_sub == c
        axes[2].scatter(X_tsne_high[mask, 0], X_tsne_high[mask, 1],
                        color=palette[c], alpha=0.4, s=12)
    axes[2].set_title('t-SNE (perplexity=50)\n[UMAP not installed — fallback]')
    axes[2].set_xlabel('t-SNE 1 (p=50)')
    axes[2].set_ylabel('t-SNE 2 (p=50)')

plt.suptitle('Three Dimensionality Reduction Techniques Compared', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
np.random.seed(42)
sns.set_theme(style='whitegrid')

# ════════════════════════════════════════════════════════════════════════════
#  Gaussian Mixture Model — Soft Assignments
#  Unlike KMeans (hard: every customer belongs to exactly one cluster),
#  GMM gives each customer a PROBABILITY for each segment.
#  Some customers are clearly in one segment; others are "on the boundary".
# ════════════════════════════════════════════════════════════════════════════

gmm = GaussianMixture(
    n_components=4,
    covariance_type='full',   # each cluster has its own covariance matrix
    n_init=5,
    random_state=42
)
gmm.fit(X_scaled)

# Get cluster probabilities: shape (N, 4)
gmm_proba  = gmm.predict_proba(X_scaled)     # probability for each cluster
gmm_labels = gmm.predict(X_scaled)           # hard assignment (argmax)

# "Uncertainty" = 1 - max probability
# High uncertainty means the customer sits on the boundary between segments
max_proba   = gmm_proba.max(axis=1)
uncertainty = 1 - max_proba

df['GMM_Cluster']   = gmm_labels
df['GMM_Certainty'] = max_proba

# Find the 10 most uncertain customers (those who could belong to multiple segments)
uncertain_mask = max_proba < 0.70
n_uncertain    = uncertain_mask.sum()
print(f'Customers with max segment probability < 0.70: {n_uncertain} ({n_uncertain/N:.1%})')
print(f'These are "boundary" customers — targeted campaigns may be less effective.\n')

# Show the 10 highest-uncertainty customers
top10_uncertain = (df[['CustomerID', 'Recency', 'Frequency', 'MonetaryValue', 'GMM_Certainty']]
                   .nsmallest(10, 'GMM_Certainty'))
print('Top 10 most uncertain customers (hardest to assign to a single segment):')
# Add cluster probabilities
proba_df = pd.DataFrame(gmm_proba, columns=[f'P(Cluster{i})' for i in range(4)])
proba_df['CustomerID'] = df['CustomerID'].values
print(top10_uncertain.merge(proba_df, on='CustomerID')
      .round(3).to_string(index=False))

In [ ]:
np.random.seed(42)
sns.set_theme(style='whitegrid')

# ── GMM certainty visualisation in PCA space ─────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: hard GMM assignments
for c in range(4):
    mask = gmm_labels == c
    axes[0].scatter(X_pca[mask, 0], X_pca[mask, 1],
                    color=palette[c], alpha=0.4, s=12,
                    label=f'Cluster {c} (n={mask.sum()})')
axes[0].set_title('GMM Hard Assignments (k=4)')
axes[0].set_xlabel('PC1')
axes[0].set_ylabel('PC2')
axes[0].legend(fontsize=8)

# Right: colour by certainty
sc = axes[1].scatter(X_pca[:, 0], X_pca[:, 1],
                     c=max_proba, cmap='RdYlGn',
                     vmin=0.5, vmax=1.0,
                     s=12, alpha=0.6)
plt.colorbar(sc, ax=axes[1], label='Max cluster probability (green=certain, red=uncertain)')
axes[1].set_title('GMM Assignment Certainty')
axes[1].set_xlabel('PC1')
axes[1].set_ylabel('PC2')

plt.suptitle('Gaussian Mixture Model — Soft Assignments', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

# Compare GMM vs KMeans using ARI
ari_gm_km = adjusted_rand_score(kmeans_labels, gmm_labels)
print(f'ARI (GMM vs KMeans): {ari_gm_km:.3f}')

In [ ]:
np.random.seed(42)
sns.set_theme(style='whitegrid')

# ════════════════════════════════════════════════════════════════════════════
#  Anomaly Detection — Flag Suspicious Customers
#  Use Isolation Forest on the full customer dataset.
#  The top 2% anomalies might be:
#    - Super VIP / whale customers (extreme high spenders)
#    - Fraudulent accounts (unusual patterns)
#    - Data entry errors
# ════════════════════════════════════════════════════════════════════════════

iso = IsolationForest(n_estimators=200, contamination=0.02, random_state=42)
iso.fit(X_scaled)
anomaly_labels  = iso.predict(X_scaled)      # +1 normal, -1 anomaly
anomaly_scores  = iso.decision_function(X_scaled)   # more negative = more anomalous

df['IsAnomaly']     = anomaly_labels == -1
df['AnomalyScore']  = anomaly_scores

n_anomalies = (anomaly_labels == -1).sum()
print(f'Isolation Forest flagged {n_anomalies} customers ({n_anomalies/N:.1%}) as anomalies.\n')

# Describe the anomalous customers
print('Profile of flagged customers:')
print(df[df['IsAnomaly']][feature_cols].describe().round(2))
print('\nProfile of normal customers:')
print(df[~df['IsAnomaly']][feature_cols].describe().round(2))

# Visualise anomalies in PCA space
fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(X_pca[~df['IsAnomaly'], 0], X_pca[~df['IsAnomaly'], 1],
           color='steelblue', alpha=0.3, s=12, label='Normal customers')
ax.scatter(X_pca[df['IsAnomaly'], 0],  X_pca[df['IsAnomaly'], 1],
           color='crimson', alpha=0.9, s=50, marker='X',
           label=f'Flagged as anomaly (n={n_anomalies})')
ax.set_title('Isolation Forest — Anomalous Customers in PCA Space', fontsize=13)
ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
np.random.seed(42)
sns.set_theme(style='whitegrid')

# ════════════════════════════════════════════════════════════════════════════
#  Business Report — Segment Summary Table
# ════════════════════════════════════════════════════════════════════════════

# Campaign and ROI recommendations per segment
campaign_recs = {
    'VIP / High-Value':     ('VIP early access, exclusive discounts, referral program, premium support', 3.5),
    'At-Risk / Dormant':    ('Win-back: 20% discount code, personalised "We miss you" email', 1.8),
    'New Customers':        ('Welcome series: onboarding emails, first-purchase incentive, tutorials', 2.2),
    'Occasional / Mid-Tier':('Upsell: cross-sell recommendations, loyalty programme invitation', 2.7)
}

# Compute per-segment statistics from KMeans assignments
report_rows = []
for c in range(4):
    mask     = df['KMeans_Cluster'] == c
    seg_name = segment_map.get(c, f'Cluster {c}')
    cam, roi = campaign_recs.get(seg_name, ('Standard campaign', 1.0))
    row = {
        'Cluster':         c,
        'Segment Name':    seg_name,
        'Size':            mask.sum(),
        'Pct of Total':    f"{mask.sum()/N:.1%}",
        'Avg Recency (d)': df.loc[mask, 'Recency'].mean().round(0).astype(int),
        'Avg Frequency':   df.loc[mask, 'Frequency'].mean().round(1),
        'Avg Monetary (£)':df.loc[mask, 'MonetaryValue'].mean().round(0).astype(int),
        'Campaign':        cam,
        'Est. ROI Multiplier': roi
    }
    report_rows.append(row)

report_df = pd.DataFrame(report_rows).set_index('Cluster')

print('=== ShopStream Customer Segmentation Business Report ===')
print()
for _, row in report_df.iterrows():
    print(f"Segment: {row['Segment Name']} (n={row['Size']}, {row['Pct of Total']})")
    print(f"  Recency: {row['Avg Recency (d)']}d | Frequency: {row['Avg Frequency']} orders | Monetary: £{row['Avg Monetary (£)']}")
    print(f"  Campaign: {row['Campaign']}")
    print(f"  Expected ROI multiplier: {row['Est. ROI Multiplier']}x baseline\n")

In [ ]:
np.random.seed(42)
sns.set_theme(style='whitegrid')

# ── Final business visualisation: 2×2 panel ──────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

seg_counts  = [report_df.loc[c, 'Size']           for c in range(4)]
seg_labels  = [report_df.loc[c, 'Segment Name']   for c in range(4)]
seg_money   = [report_df.loc[c, 'Avg Monetary (£)'] for c in range(4)]
seg_recency = [report_df.loc[c, 'Avg Recency (d)']  for c in range(4)]
seg_freq    = [report_df.loc[c, 'Avg Frequency']    for c in range(4)]

# Top-left: Segment sizes (bar chart)
bars = axes[0, 0].bar(seg_labels, seg_counts, color=palette, edgecolor='white')
axes[0, 0].set_title('Customer Count per Segment')
axes[0, 0].set_ylabel('Number of customers')
for bar, cnt in zip(bars, seg_counts):
    axes[0, 0].text(bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + 20, str(cnt), ha='center', va='bottom', fontsize=9)
axes[0, 0].tick_params(axis='x', rotation=10)

# Top-right: Average MonetaryValue per segment
axes[0, 1].bar(seg_labels, seg_money, color=palette, edgecolor='white')
axes[0, 1].set_title('Average Monetary Value per Segment (£)')
axes[0, 1].set_ylabel('£ (average annual spend)')
axes[0, 1].tick_params(axis='x', rotation=10)

# Bottom-left: Box plot of Recency per cluster
data_by_cluster = [df.loc[df['KMeans_Cluster'] == c, 'Recency'].values for c in range(4)]
bp = axes[1, 0].boxplot(data_by_cluster, labels=seg_labels, patch_artist=True,
                         medianprops=dict(color='white', linewidth=2))
for patch, col in zip(bp['boxes'], palette):
    patch.set_facecolor(col)
    patch.set_alpha(0.7)
axes[1, 0].set_title('Recency Distribution per Segment')
axes[1, 0].set_ylabel('Days since last purchase (lower = more active)')
axes[1, 0].tick_params(axis='x', rotation=10)

# Bottom-right: Box plot of Frequency per cluster
freq_by_cluster = [df.loc[df['KMeans_Cluster'] == c, 'Frequency'].values for c in range(4)]
bp2 = axes[1, 1].boxplot(freq_by_cluster, labels=seg_labels, patch_artist=True,
                          medianprops=dict(color='white', linewidth=2))
for patch, col in zip(bp2['boxes'], palette):
    patch.set_facecolor(col)
    patch.set_alpha(0.7)
axes[1, 1].set_title('Frequency Distribution per Segment')
axes[1, 1].set_ylabel('Orders per year')
axes[1, 1].tick_params(axis='x', rotation=10)

plt.suptitle('ShopStream Customer Segmentation — Executive Dashboard', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
#  Actionable Recommendations Function
#  Given a new customer's features, predict their segment and recommended action
# ════════════════════════════════════════════════════════════════════════════

def get_customer_recommendation(customer_features, kmeans_model, scaler_obj,
                                 segment_names, log_cols=None):
    """
    Predict segment and campaign recommendation for a new customer.

    Parameters
    ----------
    customer_features : list
        [Recency, Frequency, MonetaryValue, AvgBasketSize, ProductDiversity]
    kmeans_model      : fitted KMeans object
    scaler_obj        : fitted StandardScaler object
    segment_names     : dict mapping cluster_id -> segment name
    log_cols          : list of indices to log1p-transform before scaling

    Returns
    -------
    segment_id   : int
    segment_name : str
    recommendation : str
    """
    if log_cols is None:
        log_cols = [0, 1, 2]   # Recency, Frequency, MonetaryValue

    recommendations = {
        'VIP / High-Value':      'VIP Campaign: exclusive early access, premium support, referral bonus',
        'At-Risk / Dormant':     'Win-Back Campaign: 20% discount code + personalised re-engagement email',
        'New Customers':         'Onboarding: welcome series, product tutorials, first-purchase incentive',
        'Occasional / Mid-Tier': 'Upsell Campaign: cross-sell recommendations + loyalty programme invite'
    }

    feat = np.array(customer_features, dtype=float).copy()
    for i in log_cols:
        feat[i] = np.log1p(feat[i])                 # match preprocessing pipeline

    scaled      = scaler_obj.transform([feat])       # scale the single customer
    segment_id  = kmeans_model.predict(scaled)[0]    # predict cluster
    seg_name    = segment_names.get(segment_id, f'Cluster {segment_id}')
    rec         = recommendations.get(seg_name, 'Standard campaign')

    return segment_id, seg_name, rec


# ── Demo: classify 3 hypothetical new customers ──────────────────────────────
# Features: [Recency, Frequency, MonetaryValue, AvgBasketSize, ProductDiversity]
new_customers = [
    {'label': 'High-value regular',  'features': [10,  18, 850.0, 5.0, 9]},
    {'label': 'Lapsed bargain buyer', 'features': [200,  2,  45.0, 2.0, 2]},
    {'label': 'Brand new shopper',   'features': [3,    1,  92.0, 3.0, 2]},
]

print('=== Recommendation Engine Demo ===\n')
for cust in new_customers:
    seg_id, seg_name, rec = get_customer_recommendation(
        cust['features'], kmeans, scaler, segment_map
    )
    print(f"Customer profile: {cust['label']}")
    print(f"  Features: Recency={cust['features'][0]}d, "
          f"Freq={cust['features'][1]}, £{cust['features'][2]}")
    print(f"  Predicted Segment: {seg_id} — {seg_name}")
    print(f"  Recommendation: {rec}\n")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
#  Model Persistence — Save and Load the Pipeline
#  In production, save both the scaler and the clustering model together
#  so that any new batch of customers can be scored without re-fitting.
# ════════════════════════════════════════════════════════════════════════════

import os

# Bundle the pipeline into a single dictionary for clean saving
pipeline = {
    'scaler':       scaler,            # fitted StandardScaler
    'kmeans':       kmeans,            # fitted KMeans(k=4)
    'segment_map':  segment_map,       # cluster_id -> segment name
    'feature_cols': feature_cols,      # ['Recency', ...]
    'log_cols':     [0, 1, 2],         # indices to log1p before scaling
    'metadata': {
        'k': 4,
        'n_training_customers': N,
        'silhouette_score': round(silhouettes[2], 4),  # k=4 → index 2
        'trained_on': 'synthetic ShopStream data'
    }
}

save_path = 'shopstream_segmentation_pipeline.joblib'
joblib.dump(pipeline, save_path)
print(f'Pipeline saved to: {os.path.abspath(save_path)}')

# ── Demonstrate loading and using the saved pipeline ────────────────────────
loaded = joblib.load(save_path)
print(f'\nLoaded pipeline keys: {list(loaded.keys())}')
print(f'Metadata: {loaded["metadata"]}')

# Score a new batch of customers using the loaded pipeline
batch = np.array([
    [5,  22, 1100.0, 6.0, 10],   # definitely VIP
    [300, 1,   30.0, 1.5,  1],   # definitely at-risk / lapsed
])

print('\nBatch scoring with loaded pipeline:')
for feat in batch:
    s_id, s_name, rec = get_customer_recommendation(
        feat.tolist(),
        loaded['kmeans'],
        loaded['scaler'],
        loaded['segment_map'],
        loaded['log_cols']
    )
    print(f"  {feat} → Segment {s_id} ({s_name})")

In [ ]:
np.random.seed(42)
sns.set_theme(style='whitegrid')

# ── Additional profiling: feature distributions side by side per segment ─────
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.ravel()

for i, feat in enumerate(feature_cols):
    for c in range(4):
        mask = df['KMeans_Cluster'] == c
        sns.kdeplot(
            df.loc[mask, feat],
            ax=axes[i],
            color=palette[c],
            label=segment_map.get(c, f'C{c}'),
            fill=True,
            alpha=0.25
        )
    axes[i].set_title(f'Distribution: {feat}')
    axes[i].set_xlabel(feat)
    axes[i].legend(fontsize=7)

axes[5].set_visible(False)   # 5 features, 6 subplot slots
plt.suptitle('Feature Distributions Overlaid per Segment (KDE)', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
np.random.seed(42)
sns.set_theme(style='whitegrid')

# ── Silhouette diagram: visualise individual sample silhouette scores ─────────
# A silhouette score near +1 = well-matched to its cluster, far from others
# A score near 0 = on the boundary; near -1 = possibly misassigned

from sklearn.metrics import silhouette_samples

# Compute on a 1000-point sample for speed
sil_idx    = np.random.choice(N, 1000, replace=False)
X_sil      = X_scaled[sil_idx]
labels_sil = kmeans_labels[sil_idx]

sil_vals   = silhouette_samples(X_sil, labels_sil)
avg_sil    = sil_vals.mean()

fig, ax = plt.subplots(figsize=(9, 6))
y_lower = 10
for c in range(4):
    c_sil = np.sort(sil_vals[labels_sil == c])   # silhouette values for this cluster
    y_upper = y_lower + len(c_sil)
    ax.fill_betweenx(np.arange(y_lower, y_upper), 0, c_sil,
                     facecolor=palette[c], alpha=0.7)
    ax.text(-0.05, y_lower + len(c_sil) / 2,
            f'C{c}\n({segment_map.get(c,"?")})', fontsize=7, ha='right')
    y_lower = y_upper + 10

ax.axvline(avg_sil, color='crimson', linestyle='--',
           label=f'Average silhouette = {avg_sil:.3f}')
ax.set_title('Silhouette Diagram — KMeans k=4 (1,000 sample)', fontsize=13)
ax.set_xlabel('Silhouette coefficient (higher = better)')
ax.set_ylabel('Customer index (sorted within cluster)')
ax.legend()
plt.tight_layout()
plt.show()
print(f'Average silhouette score: {avg_sil:.3f} — above 0.5 indicates reasonably well-separated clusters.')

In [ ]:
np.random.seed(42)
sns.set_theme(style='whitegrid')

# ── PCA explained variance decomposition ────────────────────────────────────
# Fit full PCA (all 5 components) to understand how much each contributes
pca_full = PCA(n_components=5, random_state=42)
pca_full.fit(X_scaled)
ev_ratio = pca_full.explained_variance_ratio_

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Scree plot
axes[0].bar(range(1, 6), ev_ratio * 100, color='steelblue', edgecolor='white')
axes[0].plot(range(1, 6), np.cumsum(ev_ratio) * 100,
             'o-', color='crimson', label='Cumulative variance')
axes[0].axhline(80, color='grey', linestyle=':', label='80% threshold')
axes[0].set_title('PCA Scree Plot — Explained Variance per Component')
axes[0].set_xlabel('Principal Component')
axes[0].set_ylabel('Variance Explained (%)')
axes[0].legend()

# Feature loadings heatmap (what does each PC capture?)
loadings = pd.DataFrame(
    pca_full.components_,
    columns=feature_cols,
    index=[f'PC{i+1}' for i in range(5)]
)
sns.heatmap(loadings, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, ax=axes[1], linewidths=0.5)
axes[1].set_title('PCA Feature Loadings\n(which features does each PC capture?)')

plt.tight_layout()
plt.show()

print(f'PC1 explains {ev_ratio[0]:.1%} of variance.')
print(f'PC1+PC2 explain {sum(ev_ratio[:2]):.1%} of variance.')
print(f'PC1+PC2+PC3 explain {sum(ev_ratio[:3]):.1%} of variance.')
print('\nPC loadings tell us: which original features does each principal component mostly represent?')

## Business Summary — Plain English Segment Descriptions

The following four paragraphs are written for the Head of Marketing. No jargon.

---

**Segment 1 — VIP / High-Value Loyal Customers** (~900 customers, ~22%)  
These are your best customers. They buy frequently — around 20 orders per year — and their average annual spend is over £600. Crucially, they bought from you very recently (within the last two weeks on average). They are already deeply engaged with your brand. The risk of over-communicating is low; they welcome new product launches and exclusive access. Recommended action: give them early access to new collections, a dedicated customer success contact, and a referral programme. Retaining each of these customers is worth roughly 3.5x your average marketing investment.

---

**Segment 2 — At-Risk / Dormant Customers** (~1,100 customers, ~28%)  
These customers were once buyers — they have made 3–4 orders on average — but they haven't purchased in roughly six months. They are at real risk of churning permanently. The window to win them back is still open, but it's closing. The most effective tactic is a personalised "We miss you" email with a meaningful discount (20% works well in retail). Avoid generic newsletter blasts — these customers tune them out. Expected ROI is about 1.8x baseline if the win-back campaign is executed in the next 30 days.

---

**Segment 3 — New Customers** (~1,200 customers, ~30%)  
Your largest segment by headcount, and your biggest growth opportunity. These customers joined recently (within the last 10 days on average) and have only made 1–2 purchases. They don't yet have a strong habit of buying from ShopStream. The first 90 days are critical: customers who make a second purchase within 90 days have a 60% higher lifetime value. Recommended action: launch a 5-email welcome series over 30 days, offer a small incentive for the second purchase, and highlight your best-selling product categories. Expected ROI: 2.2x once the second purchase is secured.

---

**Segment 4 — Occasional / Mid-Tier Customers** (~800 customers, ~20%)  
Solid, reliable customers who buy every couple of months and spend around £150 per year. They like you — they just don't think of you first. They represent an upsell opportunity: data shows they rarely explore categories beyond their first purchase. A targeted cross-sell campaign — "Customers who bought X also loved Y" — is well-suited to this group. Enrol them in your loyalty programme; even modest incentives (double points on next order) can shift them towards the VIP segment over time. Expected ROI: 2.7x baseline.

## Self-Check / Lab Retrospective

Take 10 minutes to answer these before you close the notebook.

---

**Question 1:** We applied log1p transformation to MonetaryValue before scaling. What would happen if we had NOT applied this transformation? Would clustering be better or worse? Why?

<details><summary>Show answer</summary>

Without the log transform, MonetaryValue would remain heavily right-skewed with some customers spending £1,000+ and most spending £50–£200. After StandardScaling, the high spenders would have extreme z-scores (e.g., z = 10), while normal customers would cluster tightly around z = 0 or z = 1. KMeans minimises Euclidean distance, so it would be dominated by the extreme monetary outliers — effectively, the clustering would mostly separate "big spenders" from "everyone else", rather than finding the nuanced segments we care about. The log transform compresses the high end of the distribution, giving all segments more equal influence on the cluster assignments.

</details>

---

**Question 2:** The Adjusted Rand Index (ARI) between KMeans and Hierarchical Clustering was close to 1.0. Does this mean our segmentation is correct? What does and doesn't ARI tell us?

<details><summary>Show answer</summary>

High ARI tells us that **KMeans and Hierarchical agree with each other** — but it does NOT tell us that either is correct in an absolute sense. Both methods could both agree and both be wrong (they could both be discovering the same spurious structure). ARI is a measure of **consistency between two clusterings**, not ground-truth accuracy. What high ARI does give us is confidence that the structure we found is stable and not an artefact of a single algorithm's assumptions. To validate correctness, you'd want: (1) a domain expert to review the segment profiles and confirm they make business sense, (2) external validation against known outcomes (e.g., do VIP-predicted customers actually have high CLV?), or (3) an A/B test of campaigns targeting each segment.

</details>

---

**Question 3:** DBSCAN labelled some customers as noise points (label = -1). How should the marketing team treat these customers? Are they necessarily fraudulent?

<details><summary>Show answer</summary>

No — DBSCAN noise points are simply customers who don't fit neatly into any dense cluster. They could be: (a) genuinely anomalous/suspicious accounts worth investigating, (b) very unusual but legitimate customers (e.g., a single corporate buyer who orders £10,000 once a year), or (c) customers who are in transition between segments (e.g., a former VIP who is drifting towards at-risk). The marketing team should not automatically exclude noise points from campaigns. A sensible approach: cross-reference DBSCAN noise points with Isolation Forest anomaly scores. Points flagged by both warrant investigation. Points only flagged by DBSCAN can be assigned to the nearest KMeans cluster (using the distance to the centroid) and treated like any other customer in that segment.

</details>

---

**Question 4:** The GMM gave some customers a maximum cluster probability below 0.70 — we called them "boundary" customers. What is the practical business implication of low assignment certainty?

<details><summary>Show answer</summary>

Low certainty means the customer's behaviour is genuinely ambiguous — they could plausibly belong to two or more segments. This has two practical implications: (1) **campaign risk**: a highly targeted campaign for one segment (e.g., an aggressive win-back discount for at-risk customers) might be inappropriate for a boundary customer who is actually trending back towards activity — you could cheapen the relationship unnecessarily. (2) **opportunity**: boundary customers are often in transition. A customer who is 45% VIP / 45% Occasional might respond better to a "loyalty upgrade" nudge than either a pure VIP campaign or a standard newsletter. GMM allows you to design tiered campaigns: customers above 90% certainty get segment-specific campaigns; customers below 70% certainty get a gentler, broader "engagement" campaign.

</details>

## What We Built — and What's Next

### What we built in this lab

You now have a complete, end-to-end customer segmentation pipeline:

1. **Data simulation** — realistic RFM + basket features for 4,000 customers
2. **EDA** — pairplots, histograms, correlation heatmap, box plots
3. **Preprocessing** — log transforms, StandardScaler, before/after validation
4. **k selection** — elbow, silhouette, Davies-Bouldin; convergent conclusion at k=4
5. **KMeans** — fit, profile, radar charts, PCA visualisation
6. **Hierarchical clustering** — Ward dendrogram, AgglomerativeClustering, ARI comparison
7. **DBSCAN** — k-distance elbow, noise detection, three-way comparison
8. **PCA** — scree plot, loadings, 2D visualisation
9. **t-SNE** — local structure preservation, comparison with PCA
10. **UMAP** (optional) — global + local structure
11. **GMM** — soft assignments, uncertainty analysis, boundary customers
12. **Anomaly detection** — Isolation Forest on customer features
13. **Business report** — segment profiles, campaign recommendations, ROI estimates
14. **Recommendation function** — new customer prediction
15. **Model persistence** — joblib save/load for production deployment

### Week 12 Preview — Supervised Learning Fundamentals

Next week we move from *discovering* structure in data to *predicting* outcomes:

- **Linear regression** from scratch: the normal equations and gradient descent
- **Logistic regression**: the sigmoid function, log-loss, and decision boundaries
- **Regularisation**: Ridge (L2), Lasso (L1), and ElasticNet — why and when
- **Bias-variance trade-off**: the fundamental tension in supervised ML
- **Lab**: predict customer churn from the segments we identified today

The segments you created today will become the training features for next week's churn prediction model — connecting unsupervised and supervised learning into a real production pipeline.

---

*End of Notebook 12 — Week 11: Unsupervised Learning*